# install

In [1]:
# Cài đặt hoặc nâng cấp vnstock
!pip install -U vnstock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.7/278.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.7 MB/s eta 0:00:00


In [2]:
from vnstock import Quote
quote = Quote(symbol='ACB', source='KBS')


📋 Connecting Google Drive account
to save project settings.

Mounted at /content/drive


In [3]:
from vnstock import Quote
import pandas as pd

symbols = ["BID", "ACB", "VCB", "VNM", "MSN", "MWG", "HPG", "GAS", "SSI", "VRE"]

data = {}

for sym in symbols:
    quote = Quote(symbol=sym, source='KBS')
    df = quote.history(start='2024-01-01', end='2025-11-10', interval='d')
    data[sym] = df.set_index("time")["close"]

# merge thành 1 dataframe
price_df = pd.DataFrame(data)

price_df.head()

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
time,,,,,,,,,,
2024-01-02 07:00:00,34.75,14.80,55.04,57.89,68.4,40.88,18.55,64.82,22.57,22.31
2024-01-03 07:00:00,35.40,15.13,55.70,58.49,68.9,41.60,18.78,65.17,22.88,22.45
2024-01-04 07:00:00,35.28,15.32,56.63,58.49,68.1,41.60,18.75,65.77,23.34,22.60
2024-01-05 07:00:00,35.97,15.41,56.82,58.32,67.9,42.23,18.78,66.19,23.72,22.55
2024-01-08 07:00:00,37.50,15.34,57.22,57.81,66.6,41.60,18.82,65.85,23.68,22.89


# Cell 1: Tính return

In [4]:
import numpy as np

return_df = np.log(price_df / price_df.shift(1)).dropna()

return_df.head()

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
time,,,,,,,,,,
2024-01-03 07:00:00,0.018532,0.022052,0.011920,0.010311,0.007283,0.017459,0.012323,0.005385,0.013642,0.006256
2024-01-04 07:00:00,-0.003396,0.012480,0.016559,0.000000,-0.011679,0.000000,-0.001599,0.009165,0.019905,0.006659
2024-01-05 07:00:00,0.019369,0.005857,0.003349,-0.002911,-0.002941,0.015031,0.001599,0.006366,0.016150,-0.002215
2024-01-08 07:00:00,0.041656,-0.004553,0.007015,-0.008783,-0.019331,-0.015031,0.002128,-0.005150,-0.001688,0.014965
2024-01-09 07:00:00,-0.007495,-0.013784,0.011468,-0.001558,-0.007536,-0.011606,-0.005862,-0.011762,-0.002960,-0.014965


# Cell 2: Covariance matrix + plot

In [5]:
import plotly.express as px

cov_matrix = return_df.cov()

fig = px.imshow(
    cov_matrix,
    text_auto=".2e",
    color_continuous_scale="Blues",  # thang màu xanh
    title="Covariance Matrix"
)

fig.update_layout(
    template="plotly_white",
    title_font=dict(size=18),
)

fig.show()

Variance đều dương => Hợp lệ để làm input


**Xét đơn lẻ từng cổ phiếu**

Các nhóm có covar lớn: VRE, SSI, MSN, MWG
=> Nhóm này có rủi ro cao

Các nhóm có covar thấp: VCB, VNM
=> Là các cổ phiếu có mức biến độ ko cao, phù hợp làm cổ phiếu phòng thủ


**Khi đi cùng nhau**

* Nhóm cổ phiểu Bank: ACB - BID, BID - VCB, ACB - VCB
Là nhóm cùng ngành, ít đa dạng hóa (covar giảm đi ko đáng kể)

* Nhóm cổ phiếu bán lẻ: MSN, MWG
Có thể có cùng yếu tố ảnh hưởng, nhưng có cả 2 sẽ giúp giảm covar

**Ghép cổ phiểu**:

SSI khi đi với các cổ phiếu MSN, MWG, HPG đem lại hiệu quả giảm rủi ro ko tốt bằng các cổ phiếu khác, có thể là vì nhạy cảm với thị trường

**Nhìn chung**

Toàn bộ covar đều dương cho thấy có thể có 1 yếu tố chung chi phối toàn bộ các mã, khiến cho đa dạng hóa nhờ các mã này ko hiệu quả, dễ bị crash down lớn


# Cell 3: Eigen-decomposition (2x2 matrix thủ công)

In [6]:
# lấy 2 mã bất kỳ (ví dụ ACB & VCB)
sub_cov = cov_matrix.loc[["ACB", "VCB"], ["ACB", "VCB"]]

a, b = sub_cov.iloc[0, 0], sub_cov.iloc[0, 1]
c, d = sub_cov.iloc[1, 0], sub_cov.iloc[1, 1]

# characteristic equation: λ² - (a+d)λ + (ad - bc) = 0
trace = a + d
det = a*d - b*c

lambda1 = (trace + np.sqrt(trace**2 - 4*det)) / 2
lambda2 = (trace - np.sqrt(trace**2 - 4*det)) / 2

eigvals_manual = [lambda1, lambda2]

eigvals_manual

[np.float64(0.00028626704509559386), np.float64(8.48495449028329e-05)]

# Cell 4: Eigen-decomposition bằng numpy

In [7]:
eigvals_np, eigvecs_np = np.linalg.eig(sub_cov)

eigvals_np, eigvecs_np

(array([2.86267045e-04, 8.48495449e-05]),
 array([[ 0.76575877, -0.6431279 ],
        [ 0.6431279 ,  0.76575877]]))

# Cell 5: PCA thủ công



In [8]:
# chuẩn hóa dữ liệu
X = return_df - return_df.mean()

# covariance matrix
cov = np.cov(X.T)

# eigen decomposition
eigvals, eigvecs = np.linalg.eig(cov)

# sort eigenvalues giảm dần
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# project data
PC_manual = X @ eigvecs

PC_manual[:5]

,0,1,2,3,4,5,6,7,8,9
time,,,,,,,,,,
2024-01-03 07:00:00,0.037289,-0.008182,0.012336,-0.002993,0.005812,-0.003440,0.001177,0.001150,0.007124,0.000699
2024-01-04 07:00:00,0.012408,0.002187,0.015082,-0.009997,-0.001893,0.014482,-0.013099,-0.003271,0.001669,-0.013279
2024-01-05 07:00:00,0.018333,-0.010013,0.008966,-0.012179,0.005361,-0.004498,-0.007549,-0.006350,-0.006407,0.007061
2024-01-08 07:00:00,0.000705,0.015924,0.037938,-0.005728,-0.004519,-0.018868,0.005485,-0.006760,-0.020426,0.013090
2024-01-09 07:00:00,-0.023305,-0.006274,0.003739,-0.002547,-0.004521,0.000669,0.001648,0.011172,-0.011687,-0.013095


# Cell 6: PCA bằng thư viện sklearn

In [9]:
from sklearn.decomposition import PCA

pca = PCA()
PC_lib = pca.fit_transform(return_df)

PC_lib[:5]

array([[ 0.0372887 , -0.00818157, -0.01233577,  0.00299314,  0.00581247,
         0.00343951,  0.00117658,  0.00114991,  0.00712424, -0.00069935],
       [ 0.01240849,  0.0021869 , -0.01508176,  0.00999688, -0.00189301,
        -0.01448211, -0.01309902, -0.00327107,  0.0016685 ,  0.01327931],
       [ 0.01833302, -0.01001348, -0.00896642,  0.0121786 ,  0.00536136,
         0.00449817, -0.00754914, -0.00634994, -0.00640744, -0.00706115],
       [ 0.00070459,  0.01592444, -0.037938  ,  0.00572768, -0.00451924,
         0.01886753,  0.00548544, -0.0067603 , -0.02042619, -0.01308951],
       [-0.02330471, -0.00627443, -0.00373914,  0.00254721, -0.0045207 ,
        -0.00066921,  0.00164773,  0.01117179, -0.0116873 ,  0.01309477]])

# Cell 7: So sánh PCA manual vs sklearn

In [10]:
# so sánh eigenvalues
explained_var_manual = eigvals / eigvals.sum()
explained_var_lib = pca.explained_variance_ratio_

compare_df = pd.DataFrame({
    "Manual PCA": explained_var_manual,
    "Sklearn PCA": explained_var_lib
})

compare_df.head()

,Manual PCA,Sklearn PCA
0,0.519250,0.519250
1,0.126482,0.126482
2,0.066057,0.066057
3,0.063323,0.063323
4,0.053351,0.053351


Có 1 yếu tổ giải thích được 52% sự biến động (Principal Component 1 - PC1)

1 yếu tố khác giải thích được hơn 12% sự biến động (PC2)

Yếu tố 1 gần như có thể chắc chắn là Yếu tố thị trường (rủi ro hệ thống)

Khi thị trường giảm, gần như mọi cổ phiếu đều giảm và ngược lại

Yếu tố thứ 2 khả năng là ngành

# Cell 8: Phân tích PC1 và PC2

In [11]:
import pandas as pd
import plotly.graph_objects as go

# Loadings = eigenvectors
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(len(symbols))],
    index=symbols
)

# lấy PC1 & PC2
loadings_2 = loadings[["PC1", "PC2"]]

# ===== Plot =====
fig = go.Figure()

# PC1
fig.add_trace(go.Bar(
    x=loadings_2.index,
    y=loadings_2["PC1"],
    name="PC1",
    marker_color="#1f77b4"
))

# PC2
fig.add_trace(go.Bar(
    x=loadings_2.index,
    y=loadings_2["PC2"],
    name="PC2",
    marker_color="#2ca02c"
))

fig.update_layout(
    title="PCA Loadings (PC1 & PC2)",
    template="plotly_white",
    barmode="group",
    xaxis_title="Stocks",
    yaxis_title="Loading"
)

fig.show()

loadings_2


,PC1,PC2
BID,0.294024,-0.108562
ACB,0.265274,-0.116186
VCB,0.226958,-0.072665
VNM,0.213934,-0.050331
MSN,0.375665,-0.152428
MWG,0.373913,-0.163975
HPG,0.335812,-0.133816
GAS,0.209786,-0.153669
SSI,0.422757,-0.114366
VRE,0.359812,0.928536


 **PC1**

Giá trị khá đồng đều (0.2 → 0.42)

Đây gần như chắc chắn là Market Factor


**PC2**

Tất cả âm ngoại trừ VRE

Yếu tố PC2 đo chính là sự lệch pha của VRE so với thị trường

**Tổng thể**

Nhóm cổ phiếu nhạy cảm với thị trường: SSI, VRE, MWG, MSN, HPG

Nhóm cổ phiếu phòng thủ: BID, VCB, VNM, GAS

VRE là cổ phiếu đặc biệt, thuộc hệ sinh thái VIN, có ảnh hưởng mạnh đến thị trường do vốn hóa lớn, thường xuyên hút dòng tiền khỏi các cp khác

# Cell 9: Kiểm tra sự khác biệt của VRE với các mã cp khác

In [12]:
import plotly.graph_objects as go

# correlation matrix
corr_matrix = return_df.corr()

# lấy correlation của VRE với các mã khác
vre_corr = corr_matrix["VRE"].drop("VRE").sort_values(ascending=False)

vre_corr

,VRE
SSI,0.406585
HPG,0.363160
MSN,0.358473
MWG,0.347918
BID,0.337630
VCB,0.329653
ACB,0.323700
VNM,0.319957
GAS,0.221453


VRE khác biệt mạnh, có factor riêng, và nên đc dùng để đa dạng hóa